# Phase 5: Synthetic QA Generation

**Objective:** Generate Synthetic QA pairs from 500 financial articles in batches using the OpenRouter API (`google/gemma-4-31b-it:free`).

In [1]:
import os, sys, subprocess
from pathlib import Path

# Detect environment
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
            
        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel để nạp thư viện lõi (Numpy/Torch)...")
        os.kill(os.getpid(), 9) # Tự động ngắt tiến trình để ép Colab khởi động lại RAM
    else:
        print("Mã nguồn đã tồn tại. Bỏ qua cài đặt...")
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json
import time
import pandas as pd

from src.utils import resolve_path, load_config, setup_logger, ensure_dir
from src.generation import generate_synthetic_qa_batch

logger = setup_logger("SyntheticQA")
config = load_config()

# Load file .env từ Google Drive vào hệ thống Colab
if IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env') 


print("\nCell 0 complete.")

Environment: Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mã nguồn đã tồn tại. Bỏ qua cài đặt...
Project root: /content/rag-vn-finance

Cell 0 complete.


## 1. Data Loading
Load `cleaned.parquet` and sample the target 500 articles.

In [2]:
data_path = resolve_path(config['data'], 'processed_path')

# config.yaml đã trỏ trực tiếp tới file cleaned.parquet, nên không cần nối thêm tên file nữa
cleaned_file = Path(data_path)

if not cleaned_file.exists():
    # Dành cho trường hợp chạy local mà data_path là đường dẫn tương đối (data/processed/...)
    cleaned_file = REPO_ROOT / data_path

df = pd.read_parquet(cleaned_file)
logger.info(f"Loaded {len(df)} articles from {cleaned_file}")

# Sample 500 target articles
target_articles = 500
if len(df) > target_articles:
    df_sample = df.sample(n=target_articles, random_state=42).reset_index(drop=True)
else:
    df_sample = df.reset_index(drop=True)

logger.info(f"Sampled {len(df_sample)} articles for QA generation.")

[2026-05-12 02:22:43] [INFO] SyntheticQA: Loaded 9999 articles from /content/drive/MyDrive/rag-vn-finance/data/processed/cleaned.parquet
INFO:SyntheticQA:Loaded 9999 articles from /content/drive/MyDrive/rag-vn-finance/data/processed/cleaned.parquet
[2026-05-12 02:22:44] [INFO] SyntheticQA: Sampled 500 articles for QA generation.
INFO:SyntheticQA:Sampled 500 articles for QA generation.


## 2. Prompt Definition
Define the system prompt instructing the LLM to output exactly 2 QA pairs per article in a strict JSON format.

In [3]:
SYSTEM_PROMPT = """You are an expert Vietnamese financial analyst.
Your task is to read the provided financial articles and generate EXACTLY 2 Question-Answer pairs for EACH article based on its content.
The output MUST be in strictly valid JSON format following this schema:
{
  "qa_pairs": [
    {
      "doc_id": "the doc_id of the article",
      "question": "Câu hỏi tiếng Việt?",
      "answer": "Câu trả lời tiếng Việt chi tiết và chính xác dựa trên bài báo."
    }
  ]
}
Do not include any other text or markdown outside the JSON object.
"""

def build_prompt(article_batch):
    prompt = SYSTEM_PROMPT + "\n\nHere are the articles:\n"
    for i, row in article_batch.iterrows():
        prompt += f"--- Article ---\n"
        prompt += f"Doc ID: {row['doc_id']}\n"
        prompt += f"Title: {row['title']}\n"
        # Truncate content to avoid exceeding context limits
        prompt += f"Content: {row['content'][:1500]}...\n\n"
    return prompt

## 3. Batch Generation Loop
Send batches of 10 articles to OpenRouter, with exponential backoff on failure, and checkpoint every 5 batches.

In [ ]:
if IN_COLAB:
    out_dir = Path("/content/drive/MyDrive/rag-vn-finance/synthetic_qa")
else:
    out_dir = Path("synthetic_qa")
    
ensure_dir(out_dir)
raw_out_file = out_dir / "qa_pairs_raw.jsonl"
checkpoint_file = out_dir / "generation_checkpoint.json"

batch_size = 10
total_batches = (len(df_sample) + batch_size - 1) // batch_size

# Load checkpoint if exists
start_batch = 0
if checkpoint_file.exists():
    with open(checkpoint_file, "r") as f:
        chkpt = json.load(f)
        start_batch = chkpt.get("last_completed_batch", -1) + 1
        logger.info(f"Resuming from batch {start_batch}")

for b in range(start_batch, total_batches):
    logger.info(f"Processing batch {b+1}/{total_batches}")
    batch_df = df_sample.iloc[b*batch_size : (b+1)*batch_size]
    
    prompt = build_prompt(batch_df)
    
    try:
        # This function internally handles up to 2 retries with exponential backoff
        result = generate_synthetic_qa_batch(prompt)
        pairs = result.get("qa_pairs", [])
        
        # Save to raw jsonl immediately
        with open(raw_out_file, "a", encoding="utf-8") as f:
            for pair in pairs:
                f.write(json.dumps(pair, ensure_ascii=False) + "\n")
                
    except Exception as e:
        logger.error(f"Batch {b} failed entirely: {e}")
        break  # Stop execution to allow manual inspection
        
    # Checkpoint every 5 batches or on the last batch
    if (b + 1) % 5 == 0 or (b + 1) == total_batches:
        with open(checkpoint_file, "w") as f:
            json.dump({"last_completed_batch": b}, f)
        logger.info(f"Checkpoint saved at batch {b}")
        
    time.sleep(1) # Base delay to respect limits

✅ Đã ép đổi model thành: poolside/laguna-xs.2:free


[2026-05-12 02:22:50] [INFO] SyntheticQA: Resuming from batch 20
INFO:SyntheticQA:Resuming from batch 20
[2026-05-12 02:22:50] [INFO] SyntheticQA: Processing batch 21/50
INFO:SyntheticQA:Processing batch 21/50
[2026-05-12 02:23:06] [INFO] SyntheticQA: Processing batch 22/50
INFO:SyntheticQA:Processing batch 22/50
[2026-05-12 02:23:25] [INFO] SyntheticQA: Processing batch 23/50
INFO:SyntheticQA:Processing batch 23/50
[2026-05-12 02:23:44] [INFO] SyntheticQA: Processing batch 24/50
INFO:SyntheticQA:Processing batch 24/50
[2026-05-12 02:24:01] [INFO] SyntheticQA: Processing batch 25/50
INFO:SyntheticQA:Processing batch 25/50
[2026-05-12 02:24:21] [INFO] SyntheticQA: Checkpoint saved at batch 24
INFO:SyntheticQA:Checkpoint saved at batch 24
[2026-05-12 02:24:22] [INFO] SyntheticQA: Processing batch 26/50
INFO:SyntheticQA:Processing batch 26/50
[2026-05-12 02:24:36] [INFO] SyntheticQA: Processing batch 27/50
INFO:SyntheticQA:Processing batch 27/50
[2026-05-12 02:24:50] [INFO] SyntheticQA: P

## 4. Filtering & Post-processing
Remove invalid `doc_id`s, overly short questions/answers, and exact duplicate questions.

In [6]:
if raw_out_file.exists():
    raw_pairs = []
    with open(raw_out_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                # Chỉ lấy những item đúng chuẩn là Dictionary và có chứa cột doc_id
                if isinstance(item, dict) and 'doc_id' in item and 'question' in item:
                    raw_pairs.append(item)
            except json.JSONDecodeError:
                pass # Bỏ qua các dòng bị hỏng định dạng json
            
    df_qa = pd.DataFrame(raw_pairs)
    logger.info(f"Loaded {len(df_qa)} raw QA pairs")
    
    # Filter valid doc_id
    valid_doc_ids = set(df_sample['doc_id'])
    df_filtered = df_qa[df_qa['doc_id'].isin(valid_doc_ids)].copy()
    
    # Remove short Q/A
    df_filtered = df_filtered[
        (df_filtered['question'].str.len() > 15) & 
        (df_filtered['answer'].str.len() > 20)
    ]
    
    # Remove duplicates
    df_filtered = df_filtered.drop_duplicates(subset=['question'])
    
    logger.info(f"Filtered down to {len(df_filtered)} valid QA pairs")
    
    filtered_out_file = out_dir / "qa_pairs_filtered.parquet"
    df_filtered.to_parquet(filtered_out_file, index=False)
else:
    logger.warning("Raw output file not found. Run generation loop first.")

[2026-05-12 02:38:05] [INFO] SyntheticQA: Loaded 984 raw QA pairs
INFO:SyntheticQA:Loaded 984 raw QA pairs
[2026-05-12 02:38:05] [INFO] SyntheticQA: Filtered down to 983 valid QA pairs
INFO:SyntheticQA:Filtered down to 983 valid QA pairs


## 5. Train/Test Split
Split the data into 80% training set and 20% test set (immutable benchmark).

In [7]:
if 'df_filtered' in locals():
    # Split 80/20
    df_train = df_filtered.sample(frac=0.8, random_state=42)
    df_test = df_filtered.drop(df_train.index)
    
    train_file = out_dir / "qa_pairs_train.parquet"
    test_file = out_dir / "qa_pairs_test.parquet"
    
    df_train.to_parquet(train_file, index=False)
    df_test.to_parquet(test_file, index=False)
    
    logger.info(f"Saved {len(df_train)} to train, {len(df_test)} to test.")
    
    # Save stats
    stats = {
        "target_articles": target_articles,
        "raw_pairs": len(df_qa),
        "filtered_pairs": len(df_filtered),
        "train_size": len(df_train),
        "test_size": len(df_test)
    }
    with open(out_dir / "generation_stats.json", "w") as f:
        json.dump(stats, f, indent=4)
    logger.info("Phase 5 complete! Stats saved.")

[2026-05-12 02:40:22] [INFO] SyntheticQA: Saved 786 to train, 197 to test.
INFO:SyntheticQA:Saved 786 to train, 197 to test.
[2026-05-12 02:40:22] [INFO] SyntheticQA: Phase 5 complete! Stats saved.
INFO:SyntheticQA:Phase 5 complete! Stats saved.
